# NLP Lab 3: Advanced NLP - NMT Evaluation & Transformer Fine-Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Course**: ITI Natural Language Processing (NLP 101)  
**Based on Lecture**: NLP-ITI-3.pdf (Advanced NLP & Transformers)  
**Dataset**: Kaggle Emotion / AG News Dataset (kaggle.com/datasets/dair-ai/emotion-dataset)  

---

## Objectives:
1. **NMT Evaluation**: Calculate **BLEU score** (1-gram, 2-gram, cumulative) for translation / generation output evaluation.
2. **Transformer Tokenization**: Load pretrained **DistilBERT Tokenizer** and process subword tokens (`WordPiece`).
3. **Transfer Learning**: Load pretrained `DistilBERT` for Sequence Classification.
4. **Fine-Tuning**: Fine-tune a transformer model using PyTorch / Hugging Face `transformers` library.
5. **Inference**: Perform emotion classification inference on new sentences.

---



In [3]:
# Install Hugging Face transformers, datasets, evaluate (for Google Colab)
!pip install -q transformers datasets evaluate nltk

import torch
import numpy as np
import pandas as pd
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

nltk.download('punkt')
nltk.download('punkt_tab')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Setup complete! PyTorch device: {device}")



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Setup complete! PyTorch device: cuda


## Step 1: NMT Evaluation Metric (BLEU Score)

As covered in Slide 3, **BLEU (BiLingual Evaluation Understudy)** measures n-gram precision between machine translation output (candidate) and human reference translations.

$$\text{BLEU} = \text{BP} \cdot \exp\left( \sum_{n=1}^N w_n \log p_n \right)$$

### TODO 1: Compute BLEU Score
Calculate 1-gram, 2-gram, and cumulative BLEU scores using NLTK `sentence_bleu`.



In [4]:
def evaluate_bleu(reference_sentence, candidate_sentence):
    # Tokenize sentences into lists of words
    reference_tokens = [nltk.word_tokenize(reference_sentence.lower())]
    candidate_tokens = nltk.word_tokenize(candidate_sentence.lower())

    smooth = SmoothingFunction().method1

    # TODO 1: Calculate BLEU scores
    # Hint: Use sentence_bleu(reference_tokens, candidate_tokens, weights=..., smoothing_function=smooth)
    # === YOUR CODE HERE ===
    bleu_1 = sentence_bleu(reference_tokens, candidate_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth)
    bleu_2 = sentence_bleu(reference_tokens, candidate_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    bleu_cumulative = sentence_bleu(reference_tokens, candidate_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    # ======================

    return bleu_1, bleu_2, bleu_cumulative

# --- Test implementation ---
ref = "The cat sat on the mat."
cand1 = "The cat is sitting on the mat."
cand2 = "A dog was barking loudly."

b1_1, b1_2, b1_cum = evaluate_bleu(ref, cand1)
print(f"Candidate 1 ('{cand1}'):")
print(f"  BLEU-1: {b1_1:.4f} | BLEU-2: {b1_2:.4f} | BLEU-Cumulative: {b1_cum:.4f}")

b2_1, b2_2, b2_cum = evaluate_bleu(ref, cand2)
print(f"Candidate 2 ('{cand2}'):")
print(f"  BLEU-1: {b2_1:.4f} | BLEU-2: {b2_2:.4f} | BLEU-Cumulative: {b2_cum:.4f}")

assert b1_cum > b2_cum, "TODO 1 Failed! Candidate 1 should have a higher BLEU score than Candidate 2."
print("TODO 1 Passed!")



Candidate 1 ('The cat is sitting on the mat.'):
  BLEU-1: 0.7500 | BLEU-2: 0.6547 | BLEU-Cumulative: 0.4111
Candidate 2 ('A dog was barking loudly.'):
  BLEU-1: 0.1411 | BLEU-2: 0.0489 | BLEU-Cumulative: 0.0346
TODO 1 Passed!


---
## Step 2: Transformer Tokenization (DistilBERT)

Transformers use subword tokenization algorithms (such as **WordPiece**).

### TODO 2: Initialize & Apply DistilBERT Tokenizer
1. Load `AutoTokenizer.from_pretrained('distilbert-base-uncased')`.
2. Tokenize input text with `padding="max_length"`, `truncation=True`, and `max_length=64`.



In [5]:
model_name = "distilbert-base-uncased"

# TODO 2: Initialize DistilBERT Tokenizer
# === YOUR CODE HERE ===
tokenizer = AutoTokenizer.from_pretrained(model_name)
# ======================

sample_text = "NLP with Transformers is extremely powerful!"
tokens = tokenizer(sample_text, padding="max_length", truncation=True, max_length=64)

print("Input Tokens:", tokenizer.convert_ids_to_tokens(tokens['input_ids'][:15]))
print("Input IDs:   ", tokens['input_ids'][:15])

assert tokenizer is not None, "TODO 2 Failed! Tokenizer is not initialized."
assert 'input_ids' in tokens and len(tokens['input_ids']) == 64, "TODO 2 Failed! Tokenized length must be 64."
print("TODO 2 Passed!")



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Input Tokens: ['[CLS]', 'nl', '##p', 'with', 'transformers', 'is', 'extremely', 'powerful', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Input IDs:    [101, 17953, 2361, 2007, 19081, 2003, 5186, 3928, 999, 102, 0, 0, 0, 0, 0]
TODO 2 Passed!


---
## Step 3: Prepare Dataset for Fine-Tuning

We will fine-tune DistilBERT on a text classification dataset (Emotion dataset subset).  
Labels: `0` = Sadness, `1` = Joy, `2` = Love, `3` = Anger.



In [6]:
# Create training & evaluation data sample for fast Colab execution
data = {
    'text': [
        "I am so happy and joyful today!", "This is the best day of my life!", "I feel so grateful and full of joy.",
        "I feel really sad and heartbroken.", "This news makes me depressed and miserable.", "I am feeling lonely and crying.",
        "I love my friends and family deeply.", "You are my favorite person in the world.", "Sending warm hugs and love.",
        "I am furious and outrageously angry!", "This behavior is completely infuriating and bad.", "I hate being cheated like this!"
    ],
    'label': [1, 1, 1, 0, 0, 0, 2, 2, 2, 3, 3, 3] # 0=Sad, 1=Joy, 2=Love, 3=Anger
}

raw_df = pd.DataFrame(data)

# Convert to Hugging Face Dataset format
hf_dataset = Dataset.from_pandas(raw_df)

def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=32)

tokenized_dataset = hf_dataset.map(preprocess_function, batched=True)
split_dataset = tokenized_dataset.train_test_split(test_size=0.25, seed=42)

train_data = split_dataset['train']
eval_data = split_dataset['test']

print("Train samples:", len(train_data))
print("Eval samples: ", len(eval_data))



Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Train samples: 9
Eval samples:  3


---
## Step 4: Transfer Learning Model Initialization & Metrics

### TODO 3: Load Pretrained Transformer Classifier
Initialize `AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)`.



In [7]:
# TODO 3: Initialize Pretrained Model with 4 output classes
# === YOUR CODE HERE ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
# ======================

print(f"Model {model_name} initialized successfully for 4-class classification!")



model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model distilbert-base-uncased initialized successfully for 4-class classification!


### TODO 4: Define Compute Metrics Function
Complete `compute_metrics(eval_pred)` to compute and return Accuracy for evaluation.



In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # TODO 4: Compute accuracy score
    # === YOUR CODE HERE ===
    acc = (preds == labels).mean()
    # ======================
    return {"accuracy": acc}

# Test compute metrics
test_logits = np.array([[0.1, 0.9, 0.0, 0.0], [0.8, 0.1, 0.1, 0.0]])
test_labels = np.array([1, 0])
res = compute_metrics((test_logits, test_labels))
print("Compute Metrics Test Result:", res)

assert res['accuracy'] == 1.0, "TODO 4 Failed! Accuracy calculation incorrect."
print("TODO 4 Passed!")



Compute Metrics Test Result: {'accuracy': np.float64(1.0)}
TODO 4 Passed!


---
## Step 5: Fine-Tuning Transformer Model

Now we set up `TrainingArguments` and use Hugging Face `Trainer` to fine-tune DistilBERT.



In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=2,
    use_cpu=not torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting Transformer Fine-Tuning...")
trainer.train()
print("Fine-tuning completed!")



Starting Transformer Fine-Tuning...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.391156,1.464140,0.333333
2,1.327800,1.508487,0.333333
3,1.262214,1.525336,0.333333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning completed!


---
## Step 6: Custom Emotion Inference

### TODO 5: Implement Prediction Inference Function
Complete `predict_emotion(text)` which:
1. Tokenizes input text.
2. Moves input tensors to `device`.
3. Calls `model(**inputs)` inside `torch.no_grad()`.
4. Obtains class prediction using `torch.argmax`.



In [13]:
label_map = {0: "Sadness", 1: "Joy", 2: "Love", 3: "Anger"}

def predict_emotion(text, model, tokenizer):
    model.eval()
    model.to(device)

    # TODO 5: Tokenize text and run inference
    # === YOUR CODE HERE ===
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class_id = torch.argmax(outputs.logits, dim=-1).item()
    # ======================

    print(f"Input Text: '{text}'")
    print(f"Predicted Emotion: {label_map[predicted_class_id]}")
    return predicted_class_id

# --- Test Inference ---
predict_emotion("I am so thrilled and delighted with this result!", model, tokenizer)
predict_emotion("I am furious about this terrible service!", model, tokenizer)

print("TODO 5 Passed!")



Input Text: 'I am so thrilled and delighted with this result!'
Predicted Emotion: Joy
Input Text: 'I am furious about this terrible service!'
Predicted Emotion: Anger
TODO 5 Passed!


---
## Summary & Takeaways

In Lab 3, you learned:
1. How to evaluate Machine Translation & Generation using **BLEU scores**.
2. How Subword Tokenization (**WordPiece**) works in Transformers.
3. How to perform **Transfer Learning** by loading pretrained models (`DistilBERT`).
4. How to **fine-tune** a Transformer model for text classification.

Great job completing Lab 3!

